# LS-NEQ 09_rdx_sharing — RDX, Pile-Schwierigkeit und Sharing

**Dossier-Stufe J** (`lsne.experiments.STAGES["J"]`, siehe `docs/DESIGN.md` §8 und `docs/STAGES.md`).
Dieses Notebook wird von `scripts/build_notebooks.py` erzeugt; Änderungen bitte dort
vornehmen und neu generieren (`python scripts/build_notebooks.py`).

**Zweck.** `docs/DESIGN.md` §4.8 a–c: (a) **RDX** — nicht absolute Repräsentationen, sondern Differenzvektoren Δ = mean(A) − mean(B) zwischen kontrollierten Bedingungen (Kultur/Politik ↔ Proteinfehlfaltung, Jura ↔ Kultur/Politik, Englisch ↔ Deutsch, K=0 ↔ K=8) werden erklärt: welcher Anteil von ‖Δ‖² liegt in einer Kandidatenbasis (Domänenachsen, SAE-Decoderzeilen, Head-Beiträge, Hauptkomponenten), z-standardisiert gegen Zufallsbasen gleicher Größe, Stabilität über Familien; (b) der **Gegenpol wird gemessen, nicht gesetzt**: mittlere Token-NLL pro Pile-Teilgebiet aus per Range-Request gezogenen Trainingssequenzen (Schlüsselwortprofil, PubMed-Marker), Rang von `protein_misfolding`; (c) **Sharing-Index** (Jaccard der aktiven TopK-SAE-Features zwischen Bedingungen) und **Netzwerkaktivität** (Anteil aktiver Features, Masse, Participation Ratio), Kosinus der Head-Beitragsprofile.

**Eingaben.** Stimulusbank (Familiengruppen F13–F16 / F17–F18 / F19–F20 / Alltag), Richtungsbank, Domänenlexikon, SAE (`USE_SAE=True`; Rauchtest: Platzhalter), Pile per Range-Request (Cache `LSNE_CACHE/pile_nll`; `LSNE_OFFLINE=1` lässt die Rangfolge aus).

**Ausgaben.** Tabellen `pile_difficulty`, `pile_ranking`, `pile_rank_check`, `rdx`, `rdx_summary`, `rdx_stability`, `sharing`.
Alle Tabellen liegen als CSV (+ Parquet) im Laufverzeichnis
`<RUNS_ROOT>/J/<modell>@<revision>/` zusammen mit `manifest.json`
(Git-SHA, Modell, dtype, Hook, Anker-Hashes, Zeitstempel); Abbildungen unter
`<OUT_ROOT>/figures/09_rdx_sharing/`; am Ende eine kurze `summary_09_rdx_sharing_*.json` in `OUT_ROOT`.

**Ablauf.** Setup → Drive → Konfiguration → Stufe-J-Parameter → Modell → Stufe J → Schwierigkeitsrangfolge → RDX-z-Karten → Stabilität → Sharing und Aktivität → Zusammenfassung.

**Präregistrierung.** Kontrast 15 (`rdx_summary`-Zeile `contrast = culture_politics_vs_misfolding`, `basis = domain_axes`, `readout = last`: z ≥ 2 in mindestens der Hälfte der Layer, `min_effect_met`) und Kontrast 16 (`pile_rank_check`: Rang von `protein_misfolding` aus N ≥ 2 000 Sequenzen; liegt es nicht im schwersten Drittel, ersetzt `counterpole` den Gegenpol durch das tatsächlich schwerste Teilgebiet).

Hinweise: Die Zellen sind gegen Fehler abgesichert (ein fehlgeschlagener Schritt druckt
den Traceback und das Notebook läuft weiter). Mit `SMOKE = True` läuft alles in
< 3 min auf CPU mit `pythia-14m` (Rauchtest, keine wissenschaftliche Aussage).
Mit `REUSE = True` werden vorhandene Ergebnisse (manifest.json mit `status: ok`)
geladen statt neu gerechnet — so lässt sich nach einem Colab-Abbruch fortsetzen.


In [ ]:
# --- Setup: Repository holen / aktualisieren und installieren -------------------------
# Runs inside Colab (clone into /content/lsne-repo) and inside an existing checkout
# (local Jupyter, tests): a checkout is detected by lsne/__init__.py + pyproject.toml.
import importlib.util
import os
import pathlib
import subprocess
import sys

GIT_URL = os.environ.get("LSNE_GIT_URL", "https://github.com/Erikiss/Language-Switching-.git")
GIT_BRANCH = os.environ.get("LSNE_GIT_BRANCH", "claude/language-switching-non-equilibrium-ejpz0q")
REPO_DIR = os.environ.get("LSNE_REPO_DIR", "/content/lsne-repo")


def _sh(cmd: str) -> int:
    print("$", cmd, flush=True)
    return subprocess.run(cmd, shell=True).returncode


_here = pathlib.Path.cwd().resolve()
_local = next((p for p in [_here, *_here.parents]
               if (p / "lsne" / "__init__.py").exists() and (p / "pyproject.toml").exists()), None)
if _local is not None:
    REPO_DIR = str(_local)
    print("Vorhandener Checkout:", REPO_DIR)
elif pathlib.Path(REPO_DIR, ".git").exists():
    _sh(f"git -C {REPO_DIR} fetch -q --depth 1 origin {GIT_BRANCH} && "
        f"git -C {REPO_DIR} checkout -q {GIT_BRANCH} && git -C {REPO_DIR} pull -q --ff-only origin {GIT_BRANCH}")
else:
    _sh(f"git clone -q --depth 1 --branch {GIT_BRANCH} {GIT_URL} {REPO_DIR}")
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

if _local is None or importlib.util.find_spec("lsne") is None:
    _sh(f"{sys.executable} -m pip install -q -r requirements.txt")
    _sh(f"{sys.executable} -m pip install -q -e .")
import lsne
print("lsne", lsne.__version__, "aus", REPO_DIR, "| Branch", GIT_BRANCH)
print("Git-SHA:", subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())


In [ ]:
# --- Optional: TNCO (Google Research) ---------------------------------------------------
# TNCO optimises the contraction ORDER of the linearised lattice network (stage E).
# It needs the Boost headers and a C++ build (a few minutes on Colab).  Without TNCO
# lsne.tn_lattice.optimize_path falls back to opt_einsum ("dp" for <= 12 tensors,
# else "greedy"); the contracted response is identical, only the path cost / runtime
# columns of tn_paths differ ("fallback_reason" says why).  Stages A-D never need it.
INSTALL_TNCO = False   # True: bauen (nur für Notebook 04 / Stufe E sinnvoll)

try:
    import tnco
    print("tnco", getattr(tnco, "__version__", "?"), "ist bereits installiert")
except ImportError:
    if INSTALL_TNCO:
        _sh("apt-get install -y -q libboost-dev > /tmp/apt_boost.log 2>&1 || cat /tmp/apt_boost.log | tail -5")
        _sh(f'{sys.executable} -m pip install -q "tnco @ git+https://github.com/google-research/tnco"')
        try:
            import tnco
            print("tnco", getattr(tnco, "__version__", "?"), "installiert")
        except ImportError as e:
            print("TNCO konnte nicht gebaut werden:", repr(e), "-> Fallback opt_einsum (siehe docs/RUNBOOK_COLAB.md)")
    else:
        print("TNCO nicht installiert -> lsne.tn_lattice nutzt opt_einsum (dp/greedy) als Fallback.")


In [ ]:
# --- Google Drive einbinden (außerhalb von Colab wird lokal geschrieben) ---------------
import os

try:
    from google.colab import drive            # type: ignore
    drive.mount("/content/drive")
except Exception as e:                        # not on Colab, or the mount was refused
    print("Kein Colab-Drive:", repr(e).split("(")[0], "-> lokale Ausgabe")

from lsne.io import drive_root

OUT_ROOT = drive_root()                       # Colab: /content/drive/MyDrive/LSNE, sonst $LSNE_RUNS oder ./runs
RUNS_ROOT = os.path.join(OUT_ROOT, "runs")    # Laufverzeichnisse wie in jobs/shards.yaml (--out)
os.makedirs(RUNS_ROOT, exist_ok=True)
if OUT_ROOT.startswith("/content/drive"):
    # share the anchor / unigram caches between sessions (see jobs/shards.yaml defaults.env)
    os.environ.setdefault("LSNE_CACHE", os.path.join(OUT_ROOT, "cache"))
print("OUT_ROOT  =", OUT_ROOT)
print("RUNS_ROOT =", RUNS_ROOT)
print("LSNE_CACHE =", os.environ.get("LSNE_CACHE", "./cache"))


In [ ]:
# --- Konfiguration (hier anpassen) ------------------------------------------------------
NOTEBOOK = "09_rdx_sharing"
STAGE = "J"
MODEL = "EleutherAI/pythia-160m"   # HF-Modellname (State 2143: pythia-160m)
REVISION = "step98000"             # HF-Revision: step98000 / step99000 (Eingriff), step143000 (SAE)
DTYPE = "float32"                  # "float32" (Standard) | "bfloat16" (eigener Lauf, eigene Null-Dosis-Baseline)
DEVICE = "auto"                    # "auto" | "cuda" | "cpu"
FAMILIES = None                    # z. B. ["F01", "F02"] oder ["DEV01A"]; None = alle Familien (F01-F12, DEV01-DEV04)
SMOKE = False                      # True: Mini-Lauf mit pythia-14m, 1 Familie, 3 Varianten (< 3 min CPU)
SEED = 0
REUSE = True                       # vorhandene Ergebnisse (manifest.json, status ok) laden statt neu rechnen
USE_SAE = False                    # SAE-Feature-Richtungen (EleutherAI/sae-pythia-160m-32k; nur bei step143000 zertifizierbar)

from lsne.config import ModelSpec, RunConfig
from lsne.experiments import common as C

if SMOKE:
    MODEL, REVISION, DTYPE = "EleutherAI/pythia-14m", "step98000", "float32"
    FAMILIES = FAMILIES or ["DEV01"]
spec = ModelSpec(name=MODEL, revision=REVISION, dtype=DTYPE, device=DEVICE)
cfg = RunConfig(model=spec, families=FAMILIES, seed=SEED, out_root=RUNS_ROOT)
limits = C.Limits()
if SMOKE:
    cfg, limits = C.smoke_config(cfg), C.Limits.smoke()
limits.use_sae = bool(USE_SAE) and not SMOKE
limits.progress = True

_fam = f" --families {','.join(FAMILIES)}" if FAMILIES else ""
CLI_COMMAND = (f"python -m lsne.cli run --stage {STAGE} --model {MODEL} --revision {REVISION} --dtype {DTYPE}"
               f" --device {DEVICE} --out {RUNS_ROOT} --seed {SEED}{_fam}{' --smoke' if SMOKE else ''}"
               f"{' --use-sae' if limits.use_sae else ''}")
print("Modell:", spec.tag(), "| Familien:", FAMILIES or "alle", "| SMOKE:", SMOKE)
print("CLI-Äquivalent:\n  " + CLI_COMMAND)


In [ ]:
# --- Stufe-J-Parameter ---------------------------------------------------------------
J_N_PILE = None     # Pile-Sequenzen der Schwierigkeitsrangfolge (Standard 2 000 = Präregistrierung 16; Rauchtest 3)
if J_N_PILE is not None:
    limits.j_n_pile = int(J_N_PILE)
print(f"Pile-Sequenzen: {limits.j_n_pile} (je {limits.j_pile_max_len} Tokens, Range-Requests, Cache {C.cache_dir('pile_nll')}); "
      f"Zufallsbasen: {limits.j_n_random_bases}; PCA-Komponenten: {limits.j_n_pca}; Offline: {C.offline()}")


In [ ]:
# --- Modell laden -------------------------------------------------------------------
import torch
from lsne.model import load_model

mb = load_model(spec)
print(f"{spec.tag()}: {mb.n_layers} Blöcke, d_model={mb.d_model}, {mb.n_heads} Heads, "
      f"Vokabular={mb.vocab}, device={mb.device}, dtype={mb.dtype}")
print("Dossier-Eingriffsstelle (verallgemeinert):", C.dossier_site(mb).key,
      "| Tiefen-Injektion (D1b / Stufe E):", C.depth_site(mb).key)


In [ ]:
# --- Runner: Stufen direkt über run() ausführen (Ergebnisse bleiben im Speicher) --------
import contextlib
import json
import time
import traceback

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

from lsne.experiments import STAGES, STAGE_TITLES
from lsne.io import RunDir, load_df, read_manifest

ctx: dict = {}        # shared objects between stages (variants, original, direction bank)
results: dict = {}    # stage -> {table name: DataFrame}
report: dict = {}     # stage -> status / runtime / error
FIG_DIR = os.path.join(OUT_ROOT, "figures", NOTEBOOK)
os.makedirs(FIG_DIR, exist_ok=True)


def run_stage(stage: str, reuse=None, **kw):
    """Run one stage; on failure print the traceback, write error.txt and continue."""
    reuse = REUSE if reuse is None else reuse
    rd = RunDir(cfg.out_root, stage, cfg.model)
    t0 = time.perf_counter()
    if reuse and rd.manifest_path.exists():
        try:
            m = read_manifest(rd)
            if m.get("status") == "ok" and m.get("tables"):
                results[stage] = {name: load_df(rd, name) for name in m["tables"]}
                report[stage] = dict(status="reused", run_dir=str(rd.path), seconds=0.0, error="")
                print(f"Stufe {stage}: vorhandene Ergebnisse geladen aus {rd.path} "
                      f"(REUSE=False erzwingt Neuberechnung)")
                return results[stage]
        except Exception as e:  # unreadable previous run: recompute
            print("Wiederverwendung nicht möglich:", repr(e))
    print(f"Stufe {stage} ({STAGE_TITLES[stage]}) -> {rd.path}", flush=True)
    try:
        results[stage] = STAGES[stage].run(cfg, mb, rd, ctx=ctx, limits=limits, **kw)
        report[stage] = dict(status="ok", run_dir=str(rd.path), seconds=time.perf_counter() - t0, error="")
        print(f"Stufe {stage} ok in {report[stage]['seconds']:.0f} s: {', '.join(sorted(results[stage]))}")
    except Exception as e:
        traceback.print_exc()
        C.write_error(rd, stage, e, cfg, mb)
        report[stage] = dict(status="error", run_dir=str(rd.path), seconds=time.perf_counter() - t0, error=repr(e))
        results[stage] = {}
        print(f"Stufe {stage} FEHLGESCHLAGEN: {e!r} (error.txt in {rd.path})")
    return results[stage]


def table(stage: str, name: str) -> pd.DataFrame:
    """Table of a stage result or an empty DataFrame (keeps the plot cells robust)."""
    df = results.get(stage, {}).get(name)
    return df if isinstance(df, pd.DataFrame) else pd.DataFrame()


def show(df, n: int = 40):
    """display() in Jupyter / Colab, print() elsewhere."""
    if isinstance(df, pd.DataFrame):
        df = df.head(n)
    try:
        display(df)          # noqa: F821 - provided by IPython
    except NameError:
        print(df.to_string() if hasattr(df, "to_string") else df)


def savefig(name: str):
    path = os.path.join(FIG_DIR, f"{name}_{spec.tag().replace(':', '_').replace('@', '_')}.png")
    plt.gcf().savefig(path, dpi=120, bbox_inches="tight")
    print("Abbildung:", path)


@contextlib.contextmanager
def guarded(title: str):
    """Print the traceback of a failing cell section and carry on."""
    print("==", title)
    try:
        yield
    except Exception:
        traceback.print_exc()
        print(f"[{title}] übersprungen (siehe Traceback)")


In [ ]:
# --- Substufen-Bericht: die Stufen G-L fangen Fehler pro Substufe ab (docs/STAGES.md) ----
# A failed substage is recorded in manifest.json -> substages.errors (traceback in
# substage_errors.txt), its table stays empty and the stage goes on; the manifest's
# `prereg` map names the table / rows that carry every pre-registered contrast.
def substage_report(stage: str) -> dict:
    """Print the substage report and the pre-registration map of a G-L manifest."""
    rd = RunDir(cfg.out_root, stage, cfg.model)
    if not rd.manifest_path.exists():
        print(f"Stufe {stage}: kein manifest.json in {rd.path}")
        return {}
    m = read_manifest(rd)
    sub = m.get("substages") or {}
    rt = m.get("runtime_s")
    print(f"Stufe {stage}: Status {m.get('status')}" + (f", {rt:.0f} s" if isinstance(rt, (int, float)) else "")
          + f", Substufen fehlgeschlagen: {sub.get('n_failed', '?')}"
          + (" | STUB: lsne.transplant nicht importierbar" if m.get("stub") else ""))
    for name, err in (sub.get("errors") or {}).items():
        print(f"  FEHLER in Substufe {name}: {err}")
    if sub.get("n_failed"):
        print(f"  Tracebacks: {rd.path / 'substage_errors.txt'}")
    for k, v in (m.get("prereg") or {}).items():
        print(f"  Präregistrierung {k}: {v}")
    return m


def truthy(s: pd.Series) -> pd.Series:
    """Boolean view of a flag column (bool in memory, 'True'/'False' strings after a csv reload)."""
    return s.astype(str).str.lower().eq("true")


In [ ]:
# --- Stufe J ausführen ----------------------------------------------------------------
# CLI-Äquivalent:
#   python -m lsne.cli run --stage J --model EleutherAI/pythia-160m --revision step98000 --out <RUNS_ROOT> [--use-sae]
res_J = run_stage("J")
print({k: len(v) for k, v in res_J.items()})
manifest_J = substage_report("J")
if manifest_J:
    print("Kontraste (n_A, n_B):", manifest_J.get("contrasts"), "| Basen:", manifest_J.get("bases"), "| SAE:", manifest_J.get("sae_source"))


In [ ]:
# --- Kontrast 16: empirische Schwierigkeitsrangfolge der Pile-Teilgebiete (mittlere Token-NLL) ----
with guarded("Pile-Schwierigkeit"):
    rk = table("J", "pile_ranking")
    chk = table("J", "pile_rank_check")
    pile = table("J", "pile_difficulty")
    if rk.empty:
        print("Keine Rangfolge (offline oder Range-Requests fehlgeschlagen; siehe Substufen-Bericht).")
    else:
        rk = rk.sort_values("rank")
        fig, ax = plt.subplots(figsize=(7, 0.35 * len(rk) + 1.5))
        y = np.arange(len(rk))
        ax.barh(y, rk["mean_nll"], xerr=[(rk["mean_nll"] - rk["ci_low"]).clip(lower=0), (rk["ci_high"] - rk["mean_nll"]).clip(lower=0)],
                color=["C3" if s == "protein_misfolding" else "C0" for s in rk["subfield"]], capsize=2)
        ax.set_yticks(y)
        ax.set_yticklabels([f"{s} (n={n})" for s, n in zip(rk["subfield"], rk["n_seqs"])], fontsize=8)
        ax.invert_yaxis()
        ax.set_xlabel("mittlere Token-NLL [nats] (Bootstrap-CI über Sequenzen)")
        ax.set_title(f"Schwierigkeitsrangfolge aus {len(pile)} Pile-Sequenzen (rot: protein_misfolding)", fontsize=9)
        savefig("pile_difficulty_ranking")
        plt.show()
        show(chk)
        if len(chk):
            c = chk.iloc[0]
            print(f"Gegenpol laut Ersetzungsregel: {c['counterpole']} (protein_misfolding: Rang {c['rank']} von {c['n_subfields']}, "
                  f"im schwersten Drittel: {c['in_hardest_third']})")
        if "subfield_lexicon" in pile.columns:
            print("Teilgebiet (Marker) × Lexikon-Domäne derselben Sequenzen:")
            show(pd.crosstab(pile["subfield"], pile["subfield_lexicon"].fillna("")))


In [ ]:
# --- Kontrast 15: RDX — erklärter Anteil von ‖Δ‖² pro Kontrast × Basis × Layer, z gegen Zufallsbasen --
with guarded("RDX"):
    rdx = table("J", "rdx")
    rs = table("J", "rdx_summary")
    pooled = rdx[(rdx["unit"] == "pooled") & (rdx["readout"] == "last")]
    contrasts = list(dict.fromkeys(pooled["contrast"]))
    bases = [b for b in ["domain_axes", "sae", "heads", "pca"] if b in set(pooled["basis"])]
    fig, axes = plt.subplots(1, len(contrasts), figsize=(4.5 * len(contrasts), 3.6), squeeze=False)
    im = None
    for ax, cn in zip(axes[0], contrasts):
        piv = pooled[pooled["contrast"] == cn].pivot_table(index="basis", columns="layer", values="z").reindex(bases)
        im = ax.imshow(piv.to_numpy(dtype=float), cmap="viridis", aspect="auto", vmin=-2, vmax=6)
        ax.set_xticks(range(piv.shape[1]))
        ax.set_xticklabels([f"L{l}" for l in piv.columns])
        ax.set_yticks(range(piv.shape[0]))
        ax.set_yticklabels(piv.index)
        for i in range(piv.shape[0]):
            for j in range(piv.shape[1]):
                if np.isfinite(piv.iat[i, j]):
                    ax.text(j, i, f"{piv.iat[i, j]:+.1f}", ha="center", va="center", fontsize=7, color="w")
        ax.set_title(cn, fontsize=9)
    fig.suptitle("z des erklärten Anteils gegen Zufallsbasen gleicher Größe (Last-Token; z ≥ 2 = Mindesteffekt)", fontsize=10)
    if im is not None:
        fig.colorbar(im, ax=axes[0].tolist(), shrink=0.8, label="z")
    savefig("rdx_z")
    plt.show()
    show(rs.sort_values(["preregistered", "contrast", "basis"], ascending=[False, True, True]), n=60)
    if "culture_politics_vs_misfolding" not in contrasts:
        print("Hinweis: der präregistrierte Kontrast Kultur/Politik ↔ Fehlfaltung braucht die Familien F13–F16 und F19–F20 (FAMILIES = None).")
    print("Erklärter Anteil (nicht z) der Domänenachsen-Basis pro Layer:")
    show(pooled[pooled["basis"] == "domain_axes"].pivot_table(index="contrast", columns="layer", values="explained"))
    stab = table("J", "rdx_stability")
    if len(stab):
        print("Stabilität der Erklärung über Familien / Familienpaare:")
        show(stab, n=40)


In [ ]:
# --- Sharing-Index und Netzwerkaktivität (DESIGN §4.8c) -------------------------------------
with guarded("Sharing"):
    sh = table("J", "sharing")
    if sh.empty:
        print("Keine Sharing-Tabelle (kein SAE: USE_SAE=True auf step143000; im Rauchtest ein Platzhalter-SAE).")
    else:
        print("SAE-Quelle:", sh["sae_source"].iloc[0], "(synthetic_random = Platzhalter: Funktionstest, kein Befund)")
        js = sh[sh["kind"] == "sharing"]
        conds = sorted((set(js["a"]) | set(js["b"].dropna())) - {"global"})
        M = pd.DataFrame(np.nan, index=conds, columns=conds, dtype=float)
        for _, r in js.iterrows():
            if r["a"] in conds and r["b"] in conds:
                M.loc[r["a"], r["b"]] = float(r["jaccard"])
                M.loc[r["b"], r["a"]] = float(r["jaccard"])
        act = sh[sh["kind"] == "activity"].set_index("a")[["frac_active", "mean_mass", "participation_ratio"]].astype(float)
        fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
        im = a1.imshow(M.to_numpy(dtype=float), cmap="Blues", vmin=0, vmax=1)
        a1.set_xticks(range(len(conds)))
        a1.set_xticklabels(conds, rotation=45, ha="right", fontsize=8)
        a1.set_yticks(range(len(conds)))
        a1.set_yticklabels(conds, fontsize=8)
        for i in range(len(conds)):
            for j in range(len(conds)):
                if np.isfinite(M.iat[i, j]):
                    a1.text(j, i, f"{M.iat[i, j]:.2f}", ha="center", va="center", fontsize=7)
        a1.set_title("Sharing-Index S = Jaccard der aktiven TopK-Features (paarweise)", fontsize=9)
        fig.colorbar(im, ax=a1, shrink=0.8)
        (act / act.max()).plot(kind="bar", ax=a2, width=0.8)
        a2.set_title("Netzwerkaktivität A pro Bedingung (relativ zum Maximum der Spalte)", fontsize=9)
        a2.set_xlabel("Bedingung")
        a2.legend(fontsize=7)
        savefig("sharing_activity")
        plt.show()
        show(act)
        glob = js[js["a"] == "global"]
        if len(glob):
            print("Globaler Sharing-Index (alle Bedingungen):", float(glob["jaccard"].iloc[0]))
        hp = sh[sh["kind"] == "heads"]
        if len(hp):
            print("Kosinus der Head-Beitragsprofile zwischen Bedingungen (Layer-Raster):")
            show(hp.pivot_table(index=["a", "b"], columns="layer", values="cosine"), n=40)


In [ ]:
# --- Kurze Laufzusammenfassung nach OUT_ROOT schreiben ---------------------------------
with guarded("Zusammenfassung schreiben"):
    import datetime

    summary = dict(
        notebook=NOTEBOOK, stage=STAGE, model=spec.name, revision=spec.revision, dtype=spec.dtype,
        device=str(mb.device), n_layers=mb.n_layers, d_model=mb.d_model, smoke=bool(SMOKE),
        families=FAMILIES, seed=SEED, out_root=OUT_ROOT, runs_root=RUNS_ROOT, cli_command=CLI_COMMAND,
        timestamp=datetime.datetime.now(datetime.timezone.utc).isoformat(),
        report=report,
        tables={st: {n: int(len(df)) for n, df in r.items()} for st, r in results.items()},
        figures=sorted(f for f in os.listdir(FIG_DIR) if f.endswith(".png")),
    )
    path = os.path.join(OUT_ROOT, f"summary_{NOTEBOOK}_{spec.tag().replace(':', '_').replace('@', '_')}.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(C.to_jsonable(summary), f, indent=1, ensure_ascii=False)
    print("Zusammenfassung:", path)
    print(json.dumps(C.to_jsonable(summary), indent=1, ensure_ascii=False)[:3000])
